In [1]:
import yaml
import pandas as pd

with open("../config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

df_demo    = pd.read_csv(config["input_data"]["demo"])
df_web_pt1 = pd.read_csv(config["input_data"]["web_pt1"])
df_web_pt2 = pd.read_csv(config["input_data"]["web_pt2"])
df_roster  = pd.read_csv(config["input_data"]["roster"])

In [2]:
df_web_pt1.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [3]:
df_web_pt2.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58


In [4]:
df_web_pt1.columns == df_web_pt2.columns

array([ True,  True,  True,  True,  True])

In [5]:
#we merge two files without any change since they have the same exact columns
final_web_data_merged = pd.concat([df_web_pt1, df_web_pt2], ignore_index=True)
final_web_data_merged.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [6]:
final_web_data_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 755405 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   client_id     755405 non-null  int64
 1   visitor_id    755405 non-null  str  
 2   visit_id      755405 non-null  str  
 3   process_step  755405 non-null  str  
 4   date_time     755405 non-null  str  
dtypes: int64(1), str(4)
memory usage: 28.8 MB


In [7]:
#checking for missing values
final_web_data_merged.isnull().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

In [8]:
#checking for duplicates

final_web_data_merged.duplicated().sum()

np.int64(10764)

In [9]:
#standardize column names
final_web_data_merged.columns = final_web_data_merged.columns.str.lower().str.strip().str.replace(" ", "_")
final_web_data_merged.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'],
      dtype='str')

In [10]:
final_web_data_merged.dtypes

client_id       int64
visitor_id        str
visit_id          str
process_step      str
date_time         str
dtype: object

In [11]:
#Change date_time data type to datetime
final_web_data_merged["date_time"] = pd.to_datetime(final_web_data_merged["date_time"])

final_web_data_merged.dtypes

client_id                int64
visitor_id                 str
visit_id                   str
process_step               str
date_time       datetime64[us]
dtype: object

In [12]:
final_web_data_merged["date_time"].min()

Timestamp('2017-03-15 00:03:03')

In [13]:
final_web_data_merged["date_time"].max()

Timestamp('2017-06-20 23:59:57')

In [14]:
final_web_data_merged.duplicated().sum()

np.int64(10764)

In [15]:
dups = final_web_data_merged[final_web_data_merged.duplicated(keep=False)]
dups.sort_values(["client_id", "visit_id", "date_time"]).head(5)

,client_id,visitor_id,visit_id,process_step,date_time
631355,1531,934069404_34543643308,110350144_67985219545_904677,start,2017-06-02 22:32:08
631356,1531,934069404_34543643308,110350144_67985219545_904677,start,2017-06-02 22:32:08
631353,1531,934069404_34543643308,110350144_67985219545_904677,start,2017-06-02 22:32:28
631354,1531,934069404_34543643308,110350144_67985219545_904677,start,2017-06-02 22:32:28
639529,2078,585735301_21309149782,900598259_99565669243_552843,start,2017-05-06 08:29:00


In [16]:
final_web_data_merged = final_web_data_merged.drop_duplicates().reset_index(drop=True)
final_web_data_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 744641 entries, 0 to 744640
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   client_id     744641 non-null  int64         
 1   visitor_id    744641 non-null  str           
 2   visit_id      744641 non-null  str           
 3   process_step  744641 non-null  str           
 4   date_time     744641 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 28.4 MB


In [17]:
final_web_data_merged.duplicated().sum()

np.int64(0)

In [18]:
final_web_data_merged.shape

(744641, 5)

In [19]:
final_web_data_merged["process_step"].value_counts()

process_step
start      234999
step_1     162797
step_2     132750
step_3     111589
confirm    102506
Name: count, dtype: int64

In [20]:
final_web_data_merged = final_web_data_merged.sort_values(by=["client_id", "visit_id", "date_time"]).reset_index(drop=True)
final_web_data_merged

,client_id,visitor_id,visit_id,process_step,date_time
0,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36
1,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45
2,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31
3,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05
4,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09
...,...,...,...,...,...
744636,9999875,738878760_1556639849,931268933_219402947_599432,step_1,2017-06-01 22:40:08
744637,9999875,738878760_1556639849,931268933_219402947_599432,step_1,2017-06-01 22:41:28
744638,9999875,738878760_1556639849,931268933_219402947_599432,step_2,2017-06-01 22:41:47
744639,9999875,738878760_1556639849,931268933_219402947_599432,step_3,2017-06-01 22:44:58


In [21]:
final_web_data_merged.to_csv(config["output_data"]["web_merged"], index=False)